# This is the propmt chaining example as there are more than one LLM calls to get more refined final response

In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

c:\Users\mrbil\OneDrive - Higher Education Commission\Desktop\LangGraph\LangGraph-Practical\myenv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
load_dotenv(override=True)

True

In [ ]:
model = ChatOpenAI(
    model="gpt-4o", 
    api_key=os.getenv("github_OPENAI_KEY"), 
    base_url="https://models.inference.ai.azure.com" 
)



try:
    print("Testing Updated GitHub Token...")
    print("Response:", model.invoke("Hello!").content)
except Exception as e:
    print(f"Error: {e}")

Testing Updated GitHub Token...
Response: Hello! How can I assist you today? 😊


In [15]:
# Define the state 

class Blog_State(TypedDict):
    title: str
    outline: str
    content: str

In [19]:
def generate_outline(state: Blog_State) -> Blog_State:
    title = state["title"]
    prompt = f"Create a detailed outline for a blog post with the title: {title}"
    outline = model.invoke(prompt).content
    state["outline"] = outline
    return state

In [20]:
def generate_content(state: Blog_State) -> Blog_State:
    outline = state["outline"]
    prompt = f"Write a blog post based on the following outline: {outline}"
    content = model.invoke(prompt).content
    state["content"] = content
    return state

In [21]:
# define a graph

blog_workflow = StateGraph(Blog_State)

#add nodes
blog_workflow.add_node("generate_outline" , generate_outline)
blog_workflow.add_node("generate_content" , generate_content)

# add edges
blog_workflow.add_edge(START, "generate_outline")
blog_workflow.add_edge("generate_outline", "generate_content")
blog_workflow.add_edge("generate_content", END)

# compile graph
workflow = blog_workflow.compile()


In [22]:
# initial state
initial_state = {
    "title": "The Future of AI in Healthcare",
}

final_state = workflow.invoke(initial_state)

final_state

{'title': 'The Future of AI in Healthcare',
 'outline': '# Outline for Blog Post: "The Future of AI in Healthcare"\n\n---\n\n### **Introduction**\n1. **Hook:**  \n   - Start with a compelling statistic or anecdote that highlights AI\'s growing role in healthcare.  \n     *Example: "By 2030, AI in healthcare is projected to reach a market size of $194.4 billion, transforming the way care is delivered."*\n2. **Importance of the Topic:**  \n   - Discuss why AI in healthcare is a critical issue for discussion now (e.g., aging populations, increasing demand for precision medicine, global health disparities).  \n3. **Thesis Statement:**  \n   - Outline the goal of the blog post: examining how AI is reshaping healthcare today and how it might evolve in the future.\n\n---\n\n### **Section 1: Current Applications of AI in Healthcare**\n1. **Diagnostics and Imaging:**  \n   - AI\'s role in detecting diseases such as cancer, diabetic retinopathy, and cardiovascular issues.  \n   - Mention success

In [24]:
print(final_state["outline"])

# Outline for Blog Post: "The Future of AI in Healthcare"

---

### **Introduction**
1. **Hook:**  
   - Start with a compelling statistic or anecdote that highlights AI's growing role in healthcare.  
     *Example: "By 2030, AI in healthcare is projected to reach a market size of $194.4 billion, transforming the way care is delivered."*
2. **Importance of the Topic:**  
   - Discuss why AI in healthcare is a critical issue for discussion now (e.g., aging populations, increasing demand for precision medicine, global health disparities).  
3. **Thesis Statement:**  
   - Outline the goal of the blog post: examining how AI is reshaping healthcare today and how it might evolve in the future.

---

### **Section 1: Current Applications of AI in Healthcare**
1. **Diagnostics and Imaging:**  
   - AI's role in detecting diseases such as cancer, diabetic retinopathy, and cardiovascular issues.  
   - Mention successful examples like Google's DeepMind and its usage in radiology or IBM Watson.

In [25]:
print(final_state["content"])

# The Future of AI in Healthcare: Opportunities, Challenges, and What Lies Ahead

### **Introduction**

Did you know that by 2030, the global AI in healthcare market is projected to reach an astounding $194.4 billion? This growth represents more than just a technological revolution—it signals a profound transformation in how we approach healthcare. From diagnosing diseases to streamlining administrative work, artificial intelligence (AI) has already begun to reshape the landscape of medicine. 

But why talk about this now? The world is grappling with an aging population, unprecedented demands on healthcare systems, and an urgent need for precision medicine. Meanwhile, millions lack access to basic care due to global health disparities. AI holds the promise of addressing these challenges and fundamentally changing how care is delivered.

In this blog post, we dive into how AI is already making its mark in healthcare, emerging trends that will shape the field in the coming years, and the